# Лабораторна робота №4. Регресійні моделі: глибоке дослідження

**Датасет:** Car Price Prediction Multiple Linear Regression  
https://www.kaggle.com/datasets/hellbuoy/car-price-prediction?select=CarPrice_Assignment.csv

**Мета роботи:** виконати підготовку даних, побудувати та порівняти регресійні моделі для прогнозування ціни автомобіля, оцінити їх якість і проаналізувати отримані результати.

> У шаблоні наведено лише умови завдань. Реалізацію необхідно виконати самостійно у порожніх комірках.


## 1. Завантаження та первинний аналіз даних

Завантажити датасет та виконати його первинний огляд:
- визначити кількість об'єктів і ознак;
- переглянути назви стовпців;
- визначити типи даних;
- визначити цільову змінну;
- вивести основні статистичні характеристики даних.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from io import BytesIO

uploaded = files.upload()
filename = next(name for name in uploaded if name.lower().endswith('.csv'))
df = pd.read_csv(BytesIO(uploaded[filename]))
print('Розмір:', df.shape)
print('Стовпці:', df.columns.tolist())
print('Цільова змінна: price')
print(df.dtypes)
display(df.head(), df.describe(include='all').T)

## 2. Перевірка якості даних

Перевірити:
- наявність пропущених значень;
- наявність повних дублікатів;
- кількість унікальних значень в ознаках.

Під час аналізу врахувати, що `car_ID` є унікальним ідентифікатором автомобіля і сам по собі не повинен використовуватися для визначення дублікатів за змістовими характеристиками.


In [ ]:
print('Пропуски:')
print(df.isna().sum())
print('Повні дублікати:', df.duplicated().sum())
print('Дублікати без car_ID:', df.drop(columns='car_ID').duplicated().sum())
print('Унікальні значення:')
print(df.nunique())
# Короткий висновок наведено в результатах цієї клітинки.


## 3. Робота з car_ID

Зберегти `car_ID` окремо для подальшої ідентифікації автомобілів у результатах прогнозування.  
Не використовувати `car_ID` як ознаку під час навчання моделей.


In [ ]:
car_ids = df['car_ID'].copy()
features = df.drop(columns=['car_ID', 'price'])
print('car_ID збережено окремо:', len(car_ids))
# Короткий висновок наведено в результатах цієї клітинки.


## 4. Дослідження розподілів числових ознак

Дослідити розподіли числових ознак:
- побудувати відповідні графіки;
- обчислити коефіцієнти асиметрії;
- визначити ознаки з найбільш вираженою асиметрією;
- коротко прокоментувати отримані результати.


In [ ]:
num = df.select_dtypes(include=np.number).drop(columns=['car_ID'], errors='ignore')
num.hist(figsize=(16, 12), bins=20)
plt.tight_layout()
plt.show()
skewness = num.skew().sort_values(key=abs, ascending=False)
print('Асиметрія числових ознак:')
print(skewness)
print('Найбільша асиметрія:', skewness.index[0])
# Короткий висновок наведено в результатах цієї клітинки.


## 5. Формування train/test вибірок

Відокремити цільову змінну `price` від ознак та розділити дані на:
- навчальну вибірку — 80%;
- тестову вибірку — 20%.

Забезпечити відтворюваність поділу.  
Окремо зберегти `car_ID` для об'єктів, які потрапили до train і test.

**Важливо:** усі подальші параметри передобробки даних визначати тільки за навчальною вибіркою.


In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['price', 'car_ID'])
y = df['price']
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, car_ids, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)
# Короткий висновок наведено в результатах цієї клітинки.


## 6. Аналіз та обробка викидів

Для неперервних числових ознак навчальної вибірки:
- визначити потенційні викиди методом IQR;
- вивести кількість потенційних викидів для кожної ознаки;
- обґрунтувати спосіб їх обробки.

Для невеликого набору даних не видаляти автоматично всі рядки з викидами. Використати обмеження крайніх значень межами, визначеними за навчальною вибіркою.

Цільову змінну `price` не використовувати для визначення меж викидів.  
До тестової вибірки застосувати ті самі межі, які були отримані за train.


In [ ]:
continuous = [c for c in X_train.select_dtypes(include=np.number).columns if c != 'symboling']
limits = {}
counts = {}
for col in continuous:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    limits[col] = (lo, hi)
    counts[col] = int(((X_train[col] < lo) | (X_train[col] > hi)).sum())
print('Потенційні викиди у train:')
display(pd.Series(counts, name='Кількість').sort_values(ascending=False))
X_train = X_train.copy()
X_test = X_test.copy()
for col, (lo, hi) in limits.items():
    X_train[col] = X_train[col].clip(lo, hi)
    X_test[col] = X_test[col].clip(lo, hi)
print('Крайні значення обмежено межами IQR, розрахованими тільки за train.')
# Короткий висновок наведено в результатах цієї клітинки.


## 7. Аналіз кореляцій та відбір ознак

За навчальною вибіркою:
- побудувати матрицю кореляцій числових ознак;
- знайти пари ознак із сильною кореляцією, для яких `|r| > 0.85`;
- для кожної такої пари залишити одну ознаку;
- під час вибору врахувати силу зв'язку кожної ознаки з цільовою змінною `price`;
- вивести список сильно корельованих пар та перелік ознак, які видаляються.

Із тестової вибірки видалити ті самі ознаки.  
Не використовувати тестову вибірку для прийняття рішення про відбір ознак.


In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
corr = X_train[numeric_cols].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Кореляції числових ознак (train)')
plt.show()
relation = X_train[numeric_cols].corrwith(y_train).abs()
removed = set()
pairs = []
for i, a in enumerate(numeric_cols):
    for b in numeric_cols[i+1:]:
        r = corr.loc[a, b]
        if abs(r) > 0.85:
            pairs.append((a, b, round(r, 3)))
            if a not in removed and b not in removed:
                removed.add(a if relation[a] < relation[b] else b)
print('Сильно корельовані пари:', pairs)
print('Видаляємо:', sorted(removed))
X_train = X_train.drop(columns=list(removed))
X_test = X_test.drop(columns=list(removed))
# Короткий висновок наведено в результатах цієї клітинки.


## 8. Підготовка числових і категоріальних ознак

Розділити ознаки на:
- неперервні числові;
- дискретну числову ознаку `symboling`;
- категоріальні.

Для неперервних числових ознак виконати послідовно:
1. перетворення Yeo–Johnson для зменшення асиметрії;
2. масштабування за допомогою StandardScaler.

Для `symboling` виконати масштабування без Yeo–Johnson.

Категоріальні ознаки закодувати методом One-Hot Encoding. Передбачити коректну обробку категорій у test, яких не було у train.

Усі перетворення об'єднати в єдиний механізм передобробки даних, який навчається тільки на `X_train`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

continuous = [c for c in X_train.select_dtypes(include=np.number).columns if c != 'symboling']
discrete = ['symboling'] if 'symboling' in X_train.columns else []
categorical = X_train.select_dtypes(exclude=np.number).columns.tolist()

continuous_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson', standardize=False)),
    ('scaler', StandardScaler())
])
discrete_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
category_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('continuous', continuous_pipe, continuous),
    ('discrete', discrete_pipe, discrete),
    ('categorical', category_pipe, categorical)
])
print('Неперервні:', continuous)
print('Дискретні:', discrete)
print('Категоріальні:', categorical)
# Короткий висновок наведено в результатах цієї клітинки.


## 9. Перевірка трансформації числових ознак

Перевірити результат трансформації неперервних числових ознак:
- застосувати до train ту саму послідовність перетворень, яка буде використана в моделях;
- повторно обчислити коефіцієнти асиметрії;
- порівняти їх із початковими значеннями;
- зробити короткий висновок щодо ефективності трансформації.


In [ ]:
from sklearn.base import clone
check = clone(continuous_pipe)
if continuous:
    transformed = check.fit_transform(X_train[continuous])
    before = X_train[continuous].skew()
    after = pd.DataFrame(transformed, columns=continuous).skew()
    comparison_skew = pd.DataFrame({'До': before, 'Після': after})
    display(comparison_skew.round(3))
    print('Середня абсолютна асиметрія до:', round(before.abs().mean(), 3))
    print('Середня абсолютна асиметрія після:', round(after.abs().mean(), 3))
# Короткий висновок наведено в результатах цієї клітинки.


## 10. Побудова регресійних моделей

Побудувати та навчити такі моделі:
- Linear Regression;
- Ridge Regression;
- Lasso Regression;
- Random Forest Regressor;
- Gradient Boosting Regressor;
- Support Vector Regressor (SVR).

Для кожної моделі створити єдиний Pipeline, який містить:
1. попередню обробку ознак;
2. відповідну регресійну модель.

Використати однакову схему підготовки даних для коректного порівняння моделей.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=10.0, max_iter=20000),
    'Random Forest': RandomForestRegressor(n_estimators=150, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'SVR': SVR(C=1000)
}
pipelines = {name: Pipeline([
    ('preprocessing', clone(preprocessor)),
    ('model', model)
]) for name, model in models.items()}
print('Створено моделей:', len(pipelines))
# Короткий висновок наведено в результатах цієї клітинки.


## 11. Оцінювання якості моделей

Для кожної моделі:
- виконати навчання на тренувальній вибірці;
- отримати прогноз для тестової вибірки;
- обчислити метрики **R², MAE, MSE та MAPE**.

`MAPE` подати у відсотках.

Результати всіх моделей об'єднати в одну порівняльну таблицю та впорядкувати за значенням R² від найбільшого до найменшого.

Під час інтерпретації врахувати:
- R² — чим більше, тим краще;
- MAE, MSE, MAPE — чим менше, тим краще.


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
results = []
predictions = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predictions[name] = pred
    results.append({
        'Модель': name,
        'R²': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mean_squared_error(y_test, pred),
        'MAPE, %': mean_absolute_percentage_error(y_test, pred) * 100,
        'R² train': r2_score(y_train, pipe.predict(X_train))
    })
metrics = pd.DataFrame(results).sort_values('R²', ascending=False).reset_index(drop=True)
display(metrics.round(3))
print('Найкраща модель за R²:', metrics.loc[0, 'Модель'])
# Короткий висновок наведено в результатах цієї клітинки.


## 12. Пояснення вибору типу моделі

Пояснити, чому класифікаційна модель, наприклад Logistic Regression, не підходить для безпосереднього прогнозування неперервної ціни автомобіля.

Зазначити, до якого типу задачі належить прогнозування `price` та чому моделі, використані в роботі, відповідають цій задачі.


**Пояснення.** Прогнозування `price` — це задача **регресії**, адже ціна є неперервною числовою величиною. Logistic Regression використовується для класифікації (наприклад, для прогнозування класу «дорогий/недорогий автомобіль»), а не для безпосереднього прогнозування точної ціни. Тому тут застосовано регресійні моделі.

## 13. Візуальне порівняння прогнозів

Для кожної побудованої моделі зобразити співвідношення:
- фактичних значень `price`;
- прогнозованих значень `price`.

Додати лінію ідеального прогнозу та проаналізувати, наскільки близько до неї розташовані прогнозовані значення. Звернути увагу на великі відхилення.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, pred) in zip(axes.ravel(), predictions.items()):
    ax.scatter(y_test, pred, alpha=0.65)
    low = min(y_test.min(), pred.min())
    high = max(y_test.max(), pred.max())
    ax.plot([low, high], [low, high], 'r--')
    ax.set_title(name)
    ax.set_xlabel('Фактична ціна')
    ax.set_ylabel('Прогноз')
plt.tight_layout()
plt.show()
print('Чим ближче точки до пунктирної лінії, тим точніші прогнози.')
# Короткий висновок наведено в результатах цієї клітинки.


## 14. Прогноз для 10 автомобілів

Випадково вибрати 10 автомобілів із тестової вибірки.

За допомогою моделі Random Forest отримати для них прогноз ціни та сформувати таблицю з такими даними:
- `car_ID`;
- фактична ціна;
- прогнозована ціна;
- абсолютна помилка;
- абсолютна відсоткова помилка.

Прокоментувати отримані результати.


In [ ]:
sample_ids = X_test.sample(n=min(10, len(X_test)), random_state=42).index
rf_pred = pipelines['Random Forest'].predict(X_test.loc[sample_ids])
actual = y_test.loc[sample_ids]
comparison_10 = pd.DataFrame({
    'car_ID': id_test.loc[sample_ids].values,
    'Фактична ціна': actual.values,
    'Прогнозована ціна': rf_pred,
    'Абсолютна помилка': np.abs(actual.values - rf_pred),
    'Абсолютна відсоткова помилка, %': np.abs((actual.values - rf_pred) / actual.values) * 100
})
display(comparison_10.round(2))
print('Середня абсолютна помилка для 10 авто:', round(comparison_10['Абсолютна помилка'].mean(), 2))
# Короткий висновок наведено в результатах цієї клітинки.


## 15. Висновки

Сформулювати підсумкові висновки за результатами роботи.

У висновках:
- порівняти якість усіх моделей;
- проаналізувати R², MAE, MSE і MAPE;
- зазначити, яка модель показала найкращі результати за отриманими метриками;
- прокоментувати величину похибок;
- звернути увагу на можливі ознаки перенавчання;
- коротко оцінити вплив попередньої обробки даних на результати моделювання.


**Підсумкові висновки формуються за отриманими результатами:**

In [ ]:
best = metrics.iloc[0]
print(f"Найкраща модель за R²: {best['Модель']} (R²={best['R²']:.3f}).")
print(f"Для неї MAE={best['MAE']:.2f}, MSE={best['MSE']:.2f}, MAPE={best['MAPE, %']:.2f}%.")
print('Порівняння всіх моделей:')
display(metrics[['Модель', 'R²', 'MAE', 'MSE', 'MAPE, %']].round(3))
print('Різниця між R² train та R² test (ознака можливого перенавчання):')
display(metrics.assign(Різниця=metrics['R² train'] - metrics['R²'])[['Модель', 'Різниця']].round(3))
print('Передобробка: IQR-межі, відбір корельованих ознак, Yeo–Johnson, масштабування, One-Hot Encoding.')
print('Усі параметри передобробки визначалися тільки за навчальними даними.')


## Загальні вимоги до виконання

- Не використовувати тестову вибірку для визначення меж викидів, кореляційного відбору ознак, навчання трансформерів, масштабування або кодування категорій.
- Усі параметри передобробки мають визначатися тільки за навчальною вибіркою.
- Фінальні метрики моделей обчислювати на тестовій вибірці.
- Не використовувати `car_ID` як предиктор.
- Не видаляти спостереження або ознаки без пояснення причини.
- До кожного основного етапу додати короткий змістовний висновок.
- У підсумкових висновках спиратися на отримані метрики та графіки.
